In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# GPUs, Devices, and Memory

Since that section, our `Trainer` has accepted a `num_gpus` argument
and politely ignored it. For the tiny models of earlier chapters that hardly
mattered, since CPU training finished in seconds. Once models scale up,
however, device placement makes the difference between seconds and hours: a
modern desktop GPU multiplies two $4096 \times 4096$ matrices in under 2
milliseconds, whereas a CPU needs a fifth of a second or more (as we benchmark
in that section). That raw speed comes with a strict hardware
constraint: the GPU has its own dedicated memory—say 24 GB on a desktop card—and
every tensor it operates on must reside there.

Will a target model even fit? Our running 8-billion-parameter model needs 16 GB
for its weights in bf16, which fits easily for inference. Training it with Adam
in fp32, however, requires 128 GB of static state (that section)
before storing a single activation—far more than any single 24 GB or 80 GB card.
Whether a model fits on a device, and whether the GPU stays busy computing or
stalls waiting for data, comes down to where every tensor lives and how many
bytes it consumes.

In this section, we answer three core questions: how tensors and models are
placed on and copied between CPUs and GPUs; how asynchronous GPU execution
works (and how to benchmark GPU kernels accurately); and what actually fills GPU
memory during training, including how *activation checkpointing* trades extra
compute for dramatic memory savings. We then wire everything into the
device-aware `Trainer` used throughout the rest of the book. Whereas
that section and that section accounted for bytes per
parameter and per number, here we put those bytes onto real hardware (deferring
multi-GPU training to that section). Every code cell degrades
gracefully to CPU if no GPU is available—confirming correctness even if timings
require a GPU.

In [ ]:
# Silence TensorFlow's C++ info and warning messages. TensorFlow reads this
# variable when it is first imported, so this cell runs before the imports.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
import time

## Devices and Placement

### Device Objects

Every tensor lives on a specific *device*: either the CPU paired with host RAM,
or an individual GPU paired with its onboard memory. Running `nvidia-smi` in a
terminal lists every NVIDIA GPU on your machine, its total and occupied memory,
and active processes. While each framework names devices slightly differently,
all four follow one governing principle: an operation executes on a single
device, so all of its operands must live on that device.

`tf.device('/CPU:0')` and `tf.device(f'/GPU:{i}')` create device placement scopes.

We wrap device lookup in three small helpers that the rest of the book imports:
one for CPU, one for GPU $i$, and one that counts available GPUs.

In [ ]:
def cpu():
    """Get the CPU device."""
    return tf.device('/CPU:0')

def gpu(i=0):
    """Get a GPU device."""
    return tf.device(f'/GPU:{i}')

def num_gpus():
    """Get the number of available GPUs."""
    return len(tf.config.experimental.list_physical_devices('GPU'))

By default, TensorFlow uses *soft placement*, silently falling back to the CPU if a requested GPU scope does not exist.

Building on `num_gpus`, we define two fallback helpers that select a GPU when
present and fall back to CPU otherwise, allowing identical notebook code to run
unchanged on both laptops and GPU servers.

In [ ]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def try_all_gpus():
    """Return all available GPUs, or [cpu(),] if no GPU exists."""
    devices = [gpu(i) for i in range(num_gpus())]
    return devices if devices else [cpu()]

try_gpu(), try_gpu(10), try_all_gpus()

Other accelerators—such as Apple-silicon GPUs (`mps`) and Google TPUs—use their
own device identifiers, though the placement principles below apply equally to
all of them. Throughout this book we focus on CPUs and CUDA GPUs.

### Arrays on Devices

Every tensor exposes its location via its `device` attribute, while the default
creation device varies by framework.

In MXNet and PyTorch, tensors default to the CPU unless `device=` is passed;
in TensorFlow and JAX, tensors default to the first GPU when one is available
(or can be placed explicitly via `with tf.device(...):` or `jax.device_put`).
Allocating directly on the target device avoids an extra CPU-to-GPU copy.
Let's place `X` on the first GPU and `Y` on the second GPU (falling back to CPU
when fewer than two GPUs are present) and try adding them:

In [ ]:
with try_gpu():
    X = tf.ones((2, 3))
with try_gpu(1):
    Y = tf.random.uniform((2, 3))
print((X + Y).device)

Outside an explicit device scope, TensorFlow silently copies mismatched operands onto the default device (unless `tf.config.experimental.set_device_policy('explicit')` is enabled).

### Copying Between Devices

Moving tensors across devices is vastly slower than computing on them
(that section measures the gap), so silently copying
mismatched operands would hide severe performance bugs. Instead, we copy `X`
explicitly onto `Y`'s device (the figure) before adding them.

![X lives on GPU 0 and Y on GPU 1. The copy Z of X on GPU 1 (dashed) lets Y + Z run entirely on GPU 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-copyto.svg)

In [ ]:
with try_gpu(1):
    Z = tf.identity(X)  # The explicit copy: tf.identity in the target's scope
    print((Y + Z).device)  # The scope, not the operands, decides the device

Now the sum runs and lands on the second GPU alongside both operands. The
practical rule is simple: transfer data once at the boundary of your computation
(moving a tensor to a device it already occupies is essentially a no-op) and
keep every inner-loop operation on that single device.


### Models on a Device

Because a forward pass combines inputs with model parameters, the entire model
must live on the same device as its minibatches. Thanks to the module tree from
that section, every framework moves or initializes all
parameters in one call.

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Dense(1)])
with try_gpu():
    Y_hat = net(X)  # The first call creates the variables inside the scope
Y_hat

With inputs and parameters colocated on the same device, the forward pass runs
there and returns its output on that device as well.

Keep two placement rules in mind: move or initialize the model on its target
device *before* constructing the optimizer (so optimizer moments are allocated
on the same device), and allocate any new tensors created inside `forward` on
`X.device` rather than defaulting to the CPU.

## Asynchronous Execution

### The Execution Queue

A GPU does not execute operations synchronously when Python invokes them.
Instead, Python enqueues each kernel launch onto the device's command queue and
returns immediately, letting the CPU prepare subsequent work while the GPU
crunches through its queue in order. This pipelining hides Python overhead, but
it comes with a crucial timing catch: measuring how long a Python loop takes
measures *enqueueing*, not *execution*. Let's enqueue 32 products of
$4096 \times 4096$ matrices and check the wall clock twice—right after Python
finishes queuing them, and again once the device finishes computing them.

In [ ]:
if num_gpus() > 0:
    with gpu():
        A = tf.random.normal((4096, 4096))
        A @ A
    tf.test.experimental.sync_devices()  # Warm up and wait
    t = time.perf_counter()
    with gpu():
        for _ in range(32):
            B = A @ A
    print(f'time to queue 32 matrix products: {time.perf_counter() - t:.4f} sec')
    tf.test.experimental.sync_devices()
    print(f'time until they all finished:     {time.perf_counter() - t:.4f} sec')
else:
    print('No GPU: eager operations on the CPU run synchronously.')

<!-- d2l:prose id=gpus-devices-memory-md-12 fw=pytorch, tensorflow -->

GPU kernels run asynchronously while CPU operations run synchronously; `torch.cuda.synchronize()` or `tf.test.experimental.sync_devices()` waits for all queued GPU work to finish.
Python finishes enqueuing all 32 matrix multiplications in a tiny fraction of
the actual compute time. When does the CPU actually block and wait? Only when
Python needs a concrete value back on the host—printing a tensor, calling
`.item()`, converting to NumPy, or branching on a tensor in an `if` statement.
Every such *synchronization point* stalls Python until the GPU drains its queue
up to that value, leaving the GPU idle while Python catches up
(the figure). Logging `loss.item()` after every minibatch
inserts a synchronization bubble on every step—which is why our `Trainer` keeps
losses on-device inside a `d2l.Averager` and converts them to Python floats only
once per epoch (that section).

![Python queues operations op1 through op4 and continues while the GPU runs them in order. loss.item() is a synchronization point: the CPU blocks until the GPU has drained its queue; Python then resumes, and the GPU idles until it receives more work.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-async-queue.svg)

Asynchronous dispatch also dictates four rules for trustworthy GPU benchmarks:
(1) run warm-up calls first to exclude one-time allocation, library setup, and
JIT compilation; (2) synchronize the device both immediately *before* starting
the timer and immediately *before* stopping it; (3) average over multiple
repetitions; and (4) use `time.perf_counter` for sub-millisecond resolution.
(The demonstration cell above warmed up and synchronized but reported the
32-product total; our `Benchmark` helper below automates all four rules for this
and later chapters.)

In [ ]:
class Benchmark:
    """Time a callable: warmup, then device-synchronized average seconds."""
    def __init__(self, f, warmup=3, repeats=10, desc='time'):
        self.desc = desc
        for _ in range(warmup):
            f()
        tf.test.experimental.sync_devices()
        t0 = time.perf_counter()
        for _ in range(repeats):
            f()
        tf.test.experimental.sync_devices()
        self.time = (time.perf_counter() - t0) / repeats

    def __repr__(self):
        return f'{self.desc}: {1000 * self.time:.2f} ms/call'

Let's put `Benchmark` to work immediately on two claims from earlier: how much
faster a GPU is than a CPU on large matrix multiplications, and how expensive
host-to-device transfers are relative to compute. Below we time a
$4096 \times 4096$ product on CPU (fp32), on GPU in tf32 (that section)
and 16-bit precision, alongside copying a single $4096 \times 4096$ matrix from
CPU RAM to the GPU.

In [ ]:
if num_gpus() > 0:
    with cpu():
        A_cpu = tf.identity(A)  # The same matrix in main memory
    A16 = tf.cast(A, tf.bfloat16)
    def cpu_product():
        with cpu():
            return A_cpu @ A_cpu
    @tf.function  # Eager mode would reuse a cached copy of A_cpu on the GPU
    def copy_to_gpu():
        with gpu():
            return tf.identity(A_cpu)
    print(Benchmark(cpu_product, warmup=1, repeats=3,
                    desc='product on the CPU, fp32'))
    print(Benchmark(lambda: A @ A, desc='product on the GPU, tf32'))
    print(Benchmark(lambda: A16 @ A16, desc='product on the GPU, bf16'))
    print(Benchmark(copy_to_gpu, desc='copy from CPU to the GPU'))
else:
    print('No GPU: there is nothing to compare the CPU with.')

TensorFlow uses tf32 for fp32 matrix products by default.

On a modern GPU, the $4096 \times 4096$ product finishes in under 2 ms, whereas
the CPU requires 200 ms to 2 s—a $100\times$ to $1000\times$ speedup depending
on CPU core count. Dropping to 16-bit precision cuts GPU time roughly in half
again, delivering the tensor-core speedup promised in that section.

Pay special attention to the final line: merely *copying* the matrix from CPU
memory to the GPU takes three to five times *longer* than multiplying it once it
arrives—even though the multiplication executes 4096 multiply-adds per copied
element! Standard host copies first stage data through *pinned* (page-locked)
host memory so the GPU's DMA engine can pull it directly; pre-pinning batches
eliminates that extra copy and allows transfers to overlap with compute
(that section).

## GPU Memory

### Three Views of Memory

Ask how much GPU memory a training script uses and you can get three different
numbers, depending on which layer of the stack you query. Requesting memory
directly from the CUDA driver is slow, so frameworks run an internal *caching
allocator* that grabs large slabs from the driver and sub-allocates them to
tensors, retaining freed slabs for reuse rather than releasing them immediately.
This creates three nested memory views (the figure): the bytes
currently occupied by live tensors, the larger pool reserved by the framework's
allocator, and the total memory assigned by the CUDA driver to the process—the
figure displayed by `nvidia-smi`.

![The three memory views nest: the driver's allocation to the process, which nvidia-smi reports, contains the allocator's reserved blocks, which contain the live arrays.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-allocator.svg)

`tf.config.experimental.get_memory_info('GPU:0')` reports `'current'` and `'peak'` live bytes inside TensorFlow's GPU allocator.

Let's allocate a 1 GiB float32 tensor and immediately delete it to watch these
counters respond:

In [ ]:
if num_gpus() > 0:
    def report(tag):
        info = tf.config.experimental.get_memory_info('GPU:0')
        print(f'{tag}: {info["current"] / 2**20:7.1f} MiB in use, '
              f'{info["peak"] / 2**20:7.1f} MiB peak')
    with gpu():
        big = tf.zeros((256, 1024, 1024))  # 1 GiB of float32
    report('after creating big')
    del big
    report('after deleting it ')
else:
    print('No GPU: get_memory_info() tracks only GPU and TPU allocators.')

Deleting `big` drops the live-tensor byte counter by 1 GiB (in PyTorch,
TensorFlow, and JAX), yet the framework's caching allocator retains the freed
1 GiB slab in its reserved pool so `nvidia-smi` does not budge. (Never call
`empty_cache()` inside a training loop, as forcing slabs back to the CUDA
driver adds slow driver allocations on every step.)
The outermost driver view sets the hard out-of-memory ceiling, since the driver
cannot hand out more RAM than the card physically holds. To diagnose *where*
memory goes inside our code, we inspect live-byte counters—especially the *peak*
memory reached during a call, which determines whether a step succeeds or
crashes with out-of-memory (OOM).

### Memory During Training

In that section, we established the static memory floor for fp32
Adam training: $16N$ bytes for $N$ parameters ($4N$ weights, $4N$ gradients, and
$8N$ Adam moments). During an actual training step, a fourth term joins them:
*activations*. Backpropagation must retain intermediate forward-pass outputs
until the backward pass consumes them—and unlike the static $16N$, activation
memory scales linearly with minibatch size and layer width. Let's calculate it
before measuring: consider a three-hidden-layer MLP of width 4096 taking
1024-dimensional inputs ($N \approx 3.8 \times 10^7$ parameters, so
$4N \approx 144$ MiB) at minibatch size 4096. The three hidden-layer outputs are
each $4096 \times 4096$ fp32 tensors of 64 MiB—192 MiB in total, already a third
larger than the weights themselves!

In [ ]:
if num_gpus() > 0:
    start = tf.config.experimental.get_memory_info('GPU:0')['current']
    def in_use():  # Memory in use beyond that at the start of the cell
        info = tf.config.experimental.get_memory_info('GPU:0')
        return f'{(info["current"] - start) / 2**20:7.1f} MiB'
    with gpu():
        net = tf.keras.Sequential(
            [tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(10)])
        X = tf.random.normal((4096, 1024))
        y = tf.random.uniform((4096,), 0, 10, dtype=tf.int32)
        net(X)  # The first call creates the variables
    optimizer = tf.keras.optimizers.Adam()
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    print('weights           ', in_use())
    with tf.GradientTape() as tape:
        loss = loss_fn(y, net(X, training=True))
    print('+ activations     ', in_use())
    grads = tape.gradient(loss, net.trainable_variables)
    print('+ gradients       ', in_use())
    optimizer.apply_gradients(zip(grads, net.trainable_variables))
    print('+ optimizer state ', in_use())
else:
    print('No GPU: run this cell on a GPU to see the four readings.')

The readings trace the exact lifecycle of a training step: weights ($4N$ bytes)
plus inputs sit in memory first; the forward pass adds the stored activations
(~192 MiB, exceeding the weights); the backward pass consumes and frees those
activations while materializing gradients ($4N$); and the first optimizer step
allocates Adam's two moments ($8N$). From step two onward, memory oscillates
between the static $16N$ floor and the forward-pass activation peak.
Shrinking the minibatch size is the quickest lever when activations push memory
over the edge, since it cuts activation footprint without altering the
architecture. By contrast, if memory climbs steadily *from step to step*, look
for a leak keeping old computation graphs alive—the classic culprit being
appending raw loss tensors to a Python list without detaching them first.

### Activation Checkpointing

There is a second, architectural lever for activation memory. Standard
backpropagation stores every layer's intermediate output only to read it once on
the backward sweep. *Activation checkpointing* (also called *gradient
checkpointing* or *rematerialization*, [@Chen.Xu.Zhang.ea.2016]) discards
most intermediate activations during the forward pass, retaining only the inputs
at the boundaries of chosen segments, and recomputes each segment's internal
activations on the fly during the backward pass. (Despite the shared word, this
in-memory technique is unrelated to the disk checkpoints of
that section.)

Partitioning $K$ uniform layers into segments of $s$ layers stores only $K/s$
segment boundaries plus $s$ live activations inside the active segment—about
$K/s + s$ tensors instead of $K$. Choosing $s = \sqrt{K}$ drops peak activation
memory from $O(K)$ down to $O(\sqrt{K})$! The trade-off is one extra forward
pass per segment during backpropagation (the figure).
Since a linear layer's backward pass computes two matrix products (input
gradient and weight gradient) while its forward pass computes one, adding an
extra forward pass increases total step compute from roughly 3 forward-pass
equivalents to 4—about a 33% rough upper bound, depending on hardware and layer
mix.

![(a) Standard backpropagation stores the activations of every layer. (b) Checkpointing stores only the inputs of the segments (filled) and recomputes the activations inside each segment (open) during the backward pass. With $K$ layers in segments of $\sqrt{K}$ layers, stored activations fall from $O(K)$ to $O(\sqrt{K})$ at the cost of one extra forward pass.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-activation-checkpoint.svg)

Activation checkpointing shines on deep stacks of identical blocks, such as our
residual MLP (that section) and modern transformers. Below
we define `ResidualBlock` with $\mathcal{N}(0, 0.01^2)$ weights and zero biases
across all frameworks, along with a `run_stack` helper that groups blocks into
segments with optional checkpointing.

In [ ]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        normal = tf.keras.initializers.RandomNormal  # One instance per layer
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu',
                                  kernel_initializer=normal(stddev=0.01)),
            tf.keras.layers.Dense(num_hiddens,
                                  kernel_initializer=normal(stddev=0.01))])

    def call(self, X):
        return X + self.body(X)

def run_stack(blocks, X, use_checkpoint=False, segment_size=4):
    for start in range(0, len(blocks), segment_size):
        # Bind this segment's blocks now: the backward pass calls segment
        # again after the loop has moved on
        def segment(X, segment_blocks=blocks[start:start + segment_size]):
            for blk in segment_blocks:
                X = blk(X)
            return X
        f = tf.recompute_grad(segment) if use_checkpoint else segment
        X = f(X)
    return X

`tf.recompute_grad(segment)` wraps `segment` so `GradientTape` retains only its segment boundary input and recomputes internal activations on the backward sweep.

First, a sanity check: recomputing activations must never alter the math. Let's
verify on an eight-block stack split into two checkpointed segments that all
gradients match standard backpropagation.

In [ ]:
blocks = [ResidualBlock(64) for _ in range(8)]
X = tf.random.normal((32, 64))

def stack_grads(use_checkpoint):
    with tf.GradientTape() as tape:
        loss = tf.reduce_sum(run_stack(blocks, X, use_checkpoint))
    params = [v for blk in blocks for v in blk.trainable_variables]
    return tape.gradient(loss, params)

for g, g_ckpt in zip(stack_grads(False), stack_grads(True)):
    assert np.allclose(g, g_ckpt)
print('checkpointed gradients match the ordinary ones')

That correctness check runs anywhere; to measure the memory–compute trade-off,
we move to a GPU with a deeper stack of 16 blocks (width 1024, batch size 8192)
grouped into four-block segments, reporting both *step memory* (peak memory
beyond parameters and inputs) and wall-clock time via `Benchmark`.

In [ ]:
if num_gpus() > 0:
    with gpu():
        blocks = [ResidualBlock(1024) for _ in range(16)]
        X = tf.random.normal((8192, 1024))
        run_stack(blocks, X)  # Create the variables
    params = [v for blk in blocks for v in blk.trainable_variables]
    @tf.function  # One graph for each value of use_checkpoint
    def step(use_checkpoint):
        with tf.GradientTape() as tape:
            loss = tf.reduce_sum(run_stack(blocks, X, use_checkpoint))
        return tape.gradient(loss, params)
    for use_checkpoint in (False, True):
        step(use_checkpoint)  # Trace and warm up
        tf.test.experimental.sync_devices()
        start = tf.config.experimental.get_memory_info('GPU:0')['current']
        tf.config.experimental.reset_memory_stats('GPU:0')
        timing = Benchmark(lambda: step(use_checkpoint), warmup=0)
        peak = tf.config.experimental.get_memory_info('GPU:0')['peak'] - start
        print(f'checkpointing={use_checkpoint!s:5}  step memory '
              f'{peak / 2**20:5.0f} MiB, {timing.time * 1e3:5.1f} ms per step')
else:
    print('No GPU: the peak-memory comparison needs a GPU.')

Like all TensorFlow code that we time, the step is compiled with
`tf.function`.

Without checkpointing, the backward pass holds activations for all 16 blocks at
once; grouping into four-block segments keeps only four segment boundaries plus
one active segment, cutting step memory from roughly 1.1 GiB down to 0.4–0.5 GiB
for roughly 25% more compute time—slightly better than our back-of-the-envelope
33% estimate. That favorable exchange makes activation checkpointing standard
practice when training large transformers and deep vision models.

## Device-Aware Trainer

Now we can finally wire `num_gpus` into `d2l.Trainer`. Putting our rules into
practice takes just two steps: place the model parameters on the target device
before training begins, and move each minibatch to that device at the start of
every step.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
    self.save_hyperparameters()
    self.gpus = [d2l.gpu(i) for i in range(min(num_gpus, d2l.num_gpus()))]
    self.board = d2l.ProgressBoard(xlabel='epoch', ylabel='loss')

@d2l.add_to_class(d2l.Trainer)
def prepare_batch(self, batch):
    if self.gpus:
        # tf.data.Dataset emits batches on CPU. Re-wrap them inside the
        # GPU device context so subsequent ops keep their inputs on-device
        # rather than incurring an implicit copy each step.
        with self.gpus[0]:
            batch = [tf.identity(a) for a in batch]
    return batch

Using `min(num_gpus, d2l.num_gpus())` treats `num_gpus` as an upper bound rather
than a hard requirement: on CPU-only machines, `self.gpus` is empty and training
runs seamlessly on the CPU—the exact fallback behind every `Trainer(num_gpus=1)`
call throughout the rest of the book.

How much faster does a GPU actually train a small model? While our
$4096 \times 4096$ matrix product sped up by over $100\times$, an end-to-end
training loop includes data loading, Python dispatch, and smaller layers. Let's
measure a Fashion-MNIST classifier built from two 256-wide residual blocks
(that section) with $\mathcal{N}(0, 0.01^2)$
weights and zero biases.

In [ ]:
class ResMLPClassifier(d2l.Classifier):
    def __init__(self, num_hiddens=256, num_blocks=2, lr=0.1):
        super().__init__()
        self.save_hyperparameters()
        normal = tf.keras.initializers.RandomNormal  # One instance per layer
        self.net = tf.keras.Sequential([
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(num_hiddens, activation='relu',
                                  kernel_initializer=normal(stddev=0.01)),
            *[ResidualBlock(num_hiddens) for _ in range(num_blocks)],
            tf.keras.layers.Dense(10, kernel_initializer=normal(stddev=0.01))])

We compare one post-warmup epoch on Fashion-MNIST with `num_gpus=0` versus
`num_gpus=1`:

In [ ]:
data = d2l.FashionMNIST(batch_size=256)

def epoch_time(num_gpus):
    model = ResMLPClassifier()
    trainer = d2l.Trainer(max_epochs=1, num_gpus=num_gpus)
    trainer.board.display = False
    # TensorFlow runs operations on a GPU whenever there is one, and
    # num_gpus=0 only keeps the minibatches where they are: the CPU scope
    # keeps the whole run on the CPU
    with try_gpu() if num_gpus else cpu():
        trainer.fit(model, data)  # The first epoch warms up and compiles
        return Benchmark(trainer.fit_epoch, warmup=0, repeats=1).time

for gpus in (0, 1):
    print(f'num_gpus={gpus}: {epoch_time(num_gpus=gpus):.1f} sec per epoch')

The speedup is a useful reality check: on this tiny Fashion-MNIST network, a GPU
epoch runs at most a few times faster than CPU—nowhere near the $100\times$ gain
on $4096 \times 4096$ matrix multiplies. Here the largest matrix is only
$784 \times 256$, leaving the GPU vastly underutilized while data loading and
kernel-launch overhead dominate the clock. GPUs earn their keep once large
convolutions, attention heads, and wide linear layers dominate the workload—as
they do starting in the next chapter.

From a memory perspective, this small SGD model fits comfortably with gigabytes
to spare. Return to our 8-billion-parameter model, however: shrinking batch size
and enabling activation checkpointing tame activation memory, yet the 128 GB
static floor of fp32 weights, gradients, and Adam moments remains untouched—and
remember that `Trainer` cannot warn you ahead of time before hitting OOM.
Back-of-the-envelope memory budgeting remains essential, and scaling past a
single device's memory brings us to multi-GPU parallelism in
that section.

## Summary

Every tensor and model parameter resides on a specific device, and every
operation requires its operands on the same device. Because host–device
transfers are far slower than GPU compute, place models once upfront, transfer
batches at step boundaries, and keep inner loops on-device. GPUs execute
asynchronously from a command queue, meaning Python stalls only at
synchronization points (such as `.item()` or NumPy conversions) and reliable
benchmarks require warm-up, explicit synchronization, and repeated trials.
Training memory consists of static model/optimizer state plus batch-dependent
activations; activation checkpointing cuts activation memory from $O(K)$ to
$O(\sqrt{K})$ at the cost of roughly one extra forward pass. Finally, our
device-aware `Trainer` automates single-GPU placement while falling back cleanly
to CPU when no GPU is present.

## Exercises

1. [code] **Peak memory.** Take a four-layer MLP with widths 1024, 4096,
   4096, 4096 and 10 outputs, trained with Adam, as
   in that section. Using the accounting of that
   subsection, predict the peak memory of one training step at batch sizes
   64, 256, and 1024, counting weights, gradients, optimizer state, and the
   activations the backward pass keeps. Then measure the peak on a GPU.
   Where does the prediction break down, and what did it omit?

2. **Checkpointing arithmetic.** Work out the schedule of
   the figure on paper for a stack of sixteen
   `ResidualBlock`s of width 1024 at batch size 8192.
    1. Name the arrays inside one block that the backward pass must keep,
       and count their entries.
    1. Compute the activation memory held at the peak of the backward pass
       without checkpointing and with segments of one, two, four, and eight
       blocks. State how much forward computation each schedule repeats.
    1. Which segment size would you choose here, and how does it relate to
       the rule of segments of about $\sqrt{K}$ layers in
       that section?

3. [code] **Asynchronous dispatch.** ● Use the timing rules of
   that section for both parts.
    1. For $n$ from 256 to 8192, time a round trip of an $n \times n$
       matrix from CPU to GPU and back, and time one $n \times n$ matrix
       product on the device. Find the $n$ at which the two times are
       equal, extrapolating from how each time grows with $n$ if they do
       not cross in this range. Predict this crossover from your GPU's
       published host-to-device bandwidth and matrix-product throughput,
       and explain any gap between prediction and measurement.
    1. If you have two GPUs, time 1000 matrix products of two
       $4096 \times 4096$ matrices executed on one GPU, then 500 on each of
       two GPUs issued from the same loop. Measure the scaling, explain how
       asynchronous dispatch lets Python issue work to both devices, and
       identify sources of deviation from linear scaling. Guard the
       experiment with `num_gpus() >= 2`.

4. [code] **Synchronization points.** ● Consider the training run of
   `ResMLPClassifier` in that section on a GPU.
    1. List every synchronization point in one training step, reading
       `Trainer.fit_epoch`, `Trainer.prepare_batch`, and `Averager` as
       well as the model code. Classify each as unavoidable for a correct
       loop or as an avoidable instrumentation artifact, and justify each
       classification.
    1. Suppose the loop reads the loss into a Python float after every
       step. Predict from your list how much this readout slows down the
       default `ResMLPClassifier` and
       `ResMLPClassifier(num_hiddens=2048, num_blocks=8)`. Then time one
       epoch of each model with and without the readout, and explain the
       four timings.

    *Adapted from the PyTorch documentation
    [CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html).*


[Discussions](https://d2l.discourse.group/t/36758)